# Code Set Up

In [ ]:
import packages.package1.predictor_selection as ps

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import math

from scipy import stats
from math import log

In [ ]:
def trace(base, version, filt_1, filt_2):
    
    trace_1 = ps.build_trace(
        base['training_data'][filt_1][version['top_predictors']],
        {k: version['rbc']["threshold"]["case"][k] for k in version['top_predictors']},
        name="1",
        text=False,
        y_percentage=False)
    
    trace_2 = ps.build_trace(
        base['training_data'][filt_2][version['top_predictors']],
        {k: version['rbc']["threshold"]["case"][k] for k in version['top_predictors']},
        name="2",
        text=False,
        y_percentage=False)
    
    return trace_1, trace_2

In [ ]:
def trace_test(data, version, filt_1, filt_2):
    
    trace_1 = ps.build_trace(
        data[filt_1][version['top_predictors']],
        {k: version['rbc']["threshold"]["case"][k] for k in version['top_predictors']},
        name="1",
        text=False,
        y_percentage=False)
    
    trace_2 = ps.build_trace(
        data[filt_2][version['top_predictors']],
        {k: version['rbc']["threshold"]["case"][k] for k in version['top_predictors']},
        name="2",
        text=False,
        y_percentage=False)
    
    return trace_1, trace_2

In [ ]:
def graph_hist(version, trace_val_1, trace_val_2, label_1, label_2, tresh, saving_path, model, ds):
    fig, ax = plt.subplots(figsize = (10,10))
    test_bar = list(range(len(version['top_predictors'])+1))
    ax1 = ax.bar(test_bar, trace_val_1, color="tab:olive", label = label_1)
    ax2 = ax.bar(test_bar, trace_val_2, bottom = trace_val_1, color="tab:purple", label = label_2)

    ax3 = ax.vlines([tresh], 0, 0.75, transform=ax.get_xaxis_transform(), colors='r')
    ax.legend(loc="best")

    ax.set_title('Variant count per consensus score: ' + str(model) + '-' + str(ds))
    ax.set_xlabel('Consensus Score')
    ax.set_ylabel('Number of variants')

    plt.savefig('output/' + saving_path)
    
    plt.show()

In [ ]:
def reload(item, filtering, df, predictors, ks_case_ctrl_bf, binary):
    item['training_data'] = df.data[filtering]
    item['predictors'] = predictors
    print("RBC Start")
    item['ks_case_ctrl_bf'] = ks_case_ctrl_bf
    item['ordered_predictors_by_ks'] = ps.top_predictors_by_ks(item['ks_case_ctrl_bf'], n=len(item['predictors']), reverse=True, sort_by=0)
    print(len(item['ordered_predictors_by_ks']))
    item['rbc'] = ps.roc_based_classification(item['training_data'], item['predictors'], df.pathogenic, df.control, nfold=10)
    print("RBC Done")
    item['ci_data_ks'] = {k: ps.consensus_incremental_data(item['training_data'][df.is_case], item['training_data'][df.is_ctrl], v, item['rbc']["threshold"]["case"], "greater", binary) for k, v in {"ks":item['ordered_predictors_by_ks']}.items()}
    item['top_predictors'] = item['ci_data_ks']['ks']['top']
    item['training_data']['consensus_score'] = df.top_pred_cons(item['top_predictors'], item['rbc']["threshold"]["case"])
    return item

In [ ]:
import math

def matthews_correlation_coefficient(TP, TN, FP, FN):
    numerator = (TP * TN) - (FP * FN)
    denominator = math.sqrt((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN))
    
    if denominator == 0:
        return 0  # To handle the case when denominator is zero to avoid division by zero.
    else:
        mcc = numerator / denominator
        return mcc

In [ ]:
def calculate_f1_score(TP, FP, FN):
    precision = TP / (TP + FP)
    recall = TP / (TP + FN)

    if (precision + recall) == 0:
        return 0  # To handle the case when precision + recall is zero to avoid division by zero.
    else:
        f1_score = 2 * (precision * recall) / (precision + recall)
        return f1_score


In [ ]:
import random

def select_random_elements(input_list, num_elements, seed=None):
    if num_elements > len(input_list):
        print("Error: Number of elements requested exceeds the length of the list.")
        return []
    
    if seed is not None:
        random.seed(seed)
    
    random_elements = random.sample(input_list, num_elements)
    return random_elements

In [ ]:
def calculate_accuracy(tp, tn, fp, fn):
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    return accuracy

In [ ]:
def metrics(path, contr, tresh):

    i = 0
    TN = 0 
    TP = 0
    FP = 0
    FN = 0

    for a in contr:
        if i > tresh:
            FP = FP + a
        else:
            TN = TN + a
        i += 1

    i = 0
    for a in path:
        if i <= tresh:
            FN = FN + a
        else:
            TP = TP + a
        i += 1
    print(TP)
    print(TN)
    print(FP)
    print(FN)

    f1_score = calculate_f1_score(TP, FP, FN)
    mcc_score_no_cys = matthews_correlation_coefficient(TP, TN, FP, FN)
    accuracy = calculate_accuracy(TP, TN, FP, FN)
    return(f1_score, mcc_score_no_cys, accuracy)

# Creation of the Training and Test Set

In [ ]:
df = ps.DataProcessor("data/FBN1_tableS1_allmissense.tsv")
df.read_data()
df.create_filters()

In [ ]:
df_test = ps.DataProcessor("data/FBN1_tableS1_allmissense.tsv")  # Initialize a DataProcessor with the FBN1 missense variants dataset
df_test.read_data()                                              # Load the dataset into memory
df_test.create_filters()                                         # Create boolean filters (e.g., is_mis, is_ctrl, is_case, etc.)

filtering = ~df.training_sets & df.is_mis & ~df.is_denovo       # Select only missense variants that are NOT in training set and NOT de novo
test = df.data[~filtering]                                      # Extract variants NOT matching the filter (used as complementary dataset)
test = df.data[df.is_mis]                                       # Overwrites previous line: keep only missense variants from original data
df.data = df.data[filtering]                                    # Retain only training candidates in main data
df.create_filters()                                             # Recalculate filters based on updated data

index_train = select_random_elements(list(df.data[df.is_ctrl].index), 38, seed=42)  # Randomly select 38 control variant indices
df.data = df.data[~df.data.index.isin(index_train)]                                 # Remove those control variants from main training data
df.create_filters()                                                                 # Recompute filters after deletion

filtering_test = ~df_test.training_sets & df_test.is_mis       # Keep only non-training, missense variants
df_test.data = df_test.data[filtering_test]                    # Apply the filter
df_test.create_filters()                                       # Recompute filters

filtering_test_2 = (
    (df_test.is_denovo & df_test.is_case)                     # de novo variants in cases
    | (df_test.data.index.isin(index_train) & df_test.is_ctrl)  # control variants matching previously removed training indices
)
df_test.data = df_test.data[filtering_test_2]                  # Apply this final filtering to define test set
df_test.create_filters()                                       # Recompute filters

dupl_case = df.data[df.is_case][
    df.data[df.is_case].duplicated(subset=['pos(1-based)', 'aaref', 'aaalt'])
][df.is_case]                                                  # Find duplicated case variants (same position and amino acid change)
df.data = df.data.drop(dupl_case.index, errors='ignore')       # Drop them from training data

dupl_ctrl = df.data[df.is_ctrl][
    df.data[df.is_ctrl].duplicated(subset=['pos(1-based)', 'aaref', 'aaalt'])
][df.is_ctrl]                                                  # Find duplicated control variants
df.data = df.data.drop(dupl_ctrl.index, errors='ignore')       # Drop them from training data


# Training GEMVAP

In [ ]:
data, predictors, ks_case_ctrl_bf = ps.predictors_computation(df, df.is_mis, 2)
#del ks_case_ctrl_bf["MVP_rankscore"]
base = {}
base = reload(base, ~df.training_sets & df.is_mis & ~df.is_denovo, df, predictors, ks_case_ctrl_bf, True)

In [ ]:
base['ci_data_ks']['ks']['metrics']['consensuses']

In [ ]:
base['ci_data_ks']['ks']['metrics']['npreds']

In [ ]:
values = {}

for key, item in ks_case_ctrl_bf.items():
    values[key] = item[0]

values = dict(sorted(values.items(), key=lambda item: item[1], reverse=False))


In [ ]:
fig, ax = plt.subplots(figsize = (10,10), layout='constrained')
ax.barh(list(values.keys()), list(values.values()), height = 0.5, align = "center", color="orange")
        
# Set chart title and labels
#ax.set_title('KS test p-value between Pathogenic and Control for each predictor')
ax.set_xlabel('KS test value')

plt.savefig('output/KS_Test_GEMVAP2.png')

# Show chart
plt.show()

In [ ]:
data, predictors, ks_case_ctrl_bf = ps.predictors_computation(df, ~df.is_cys, 2)
cyst = {}
cyst = reload(cyst, ~df.is_cys, df, predictors, ks_case_ctrl_bf, True)

In [ ]:
#loading conserved sites and merging tables
tableS1_cons = pd.read_csv("data/FBN1_tableS1_indomconserved.tsv", sep='\t', low_memory=False, na_values=['.'])
tableS1 = df.data.join(tableS1_cons.set_index("variantvcf"), on="variantvcf")
tableS1_test = df_test.data.join(tableS1_cons.set_index("variantvcf"), on="variantvcf")
data, predictors, ks_case_ctrl_bf = ps.predictors_computation(df, ~df.is_cys & (tableS1["in_dom_conserved"] == 0), 0)
dom = {}
dom = reload(dom, ~df.training_sets & df.is_mis & ~df.is_denovo & ~df.is_cys & (tableS1["in_dom_conserved"] == 0), df, predictors, ks_case_ctrl_bf, True)

In [ ]:
dom['ci_data_ks']['ks']['metrics']['consensuses']

In [ ]:
dom['ci_data_ks']['ks']['metrics']['npreds']

In [ ]:
#Creates a pair of distribution traces for consensus predictor scores.
#Segregates the test data based on biological relevance (cysteine mutation, domain conservation).
#Allows visual or statistical comparison between pathogenic (case) and control variant subsets.

[path_1_123, ctrl_1_123] = trace_test(df_test.data, base, (df_test.is_case & ~df_test.is_ctrl), (~df_test.is_case & df_test.is_ctrl))
[path_1_1, ctrl_1_1] = trace_test(df_test.data, base, (df_test.is_case & ~df_test.is_ctrl) & df_test.is_cys, (~df_test.is_case & df_test.is_ctrl) & df_test.is_cys)
[path_1_2, ctrl_1_2] = trace_test(df_test.data, base, (df_test.is_case & ~df_test.is_ctrl) & (tableS1_test["in_dom_conserved"] == 1), (~df_test.is_case & df_test.is_ctrl) & (tableS1_test["in_dom_conserved"] == 1))
[path_1_23, ctrl_1_23] = trace_test(df_test.data, base, (df_test.is_case & ~df_test.is_ctrl) & ~df_test.is_cys, (~df_test.is_case & df_test.is_ctrl) & ~df_test.is_cys)
[path_1_3, ctrl_1_3] = trace_test(df_test.data, base, (df_test.is_case & ~df_test.is_ctrl) & (~df_test.is_cys & (tableS1_test["in_dom_conserved"] == 0)), (~df_test.is_case & df_test.is_ctrl) & (~df_test.is_cys & (tableS1_test["in_dom_conserved"] == 0)))

In [ ]:
[path_2_123, ctrl_2_123] = trace_test(df_test.data, cyst, (df_test.is_case & ~df_test.is_ctrl), (~df_test.is_case & df_test.is_ctrl))
[path_2_1, ctrl_2_1] = trace_test(df_test.data, cyst, (df_test.is_case & ~df_test.is_ctrl) & df_test.is_cys, (~df_test.is_case & df_test.is_ctrl) & df_test.is_cys)
[path_2_2, ctrl_2_2] = trace_test(df_test.data, cyst, (df_test.is_case & ~df_test.is_ctrl) & (tableS1_test["in_dom_conserved"] == 1), (~df_test.is_case & df_test.is_ctrl) & (tableS1_test["in_dom_conserved"] == 1))
[path_2_23, ctrl_2_23] = trace_test(df_test.data, cyst, (df_test.is_case & ~df_test.is_ctrl) & ~df_test.is_cys, (~df_test.is_case & df_test.is_ctrl) & ~df_test.is_cys)
[path_2_3, ctrl_2_3] = trace_test(df_test.data, cyst, (df_test.is_case & ~df_test.is_ctrl) & (~df_test.is_cys & (tableS1_test["in_dom_conserved"] == 0)), (~df_test.is_case & df_test.is_ctrl) & (~df_test.is_cys & (tableS1_test["in_dom_conserved"] == 0)))

In [ ]:
[path_3_123, ctrl_3_123] = trace_test(df_test.data, dom, (df_test.is_case & ~df_test.is_ctrl), (~df_test.is_case & df_test.is_ctrl))
[path_3_1, ctrl_3_1] = trace_test(df_test.data, dom, (df_test.is_case & ~df_test.is_ctrl) & df_test.is_cys, (~df_test.is_case & df_test.is_ctrl) & df_test.is_cys)
[path_3_2, ctrl_3_2] = trace_test(df_test.data, dom, (df_test.is_case & ~df_test.is_ctrl) & (tableS1_test["in_dom_conserved"] == 1), (~df_test.is_case & df_test.is_ctrl) & (tableS1_test["in_dom_conserved"] == 1))
[path_3_23, ctrl_3_23] = trace_test(df_test.data,dom, (df_test.is_case & ~df_test.is_ctrl) & ~df_test.is_cys, (~df_test.is_case & df_test.is_ctrl) & ~df_test.is_cys)
[path_3_3, ctrl_3_3] = trace_test(df_test.data, dom, (df_test.is_case & ~df_test.is_ctrl) & (~df_test.is_cys & (tableS1_test["in_dom_conserved"] == 0)), (~df_test.is_case & df_test.is_ctrl) & (~df_test.is_cys & (tableS1_test["in_dom_conserved"] == 0)))

In [ ]:
i = 0

results_per_pred_base = pd.DataFrame([predictors])
results_per_pred_base = results_per_pred_base.T
results_per_pred_base.index = results_per_pred_base[0]
results_per_pred_base[0] = 0
results_per_pred_base[1] = 0
results_per_pred_base[2] = 0

results_per_pred_3 = pd.DataFrame([predictors])
results_per_pred_3 = results_per_pred_3.T
results_per_pred_3.index = results_per_pred_3[0]
results_per_pred_3[0] = 0
results_per_pred_3[1] = 0
results_per_pred_3[2] = 0

df = ps.DataProcessor("data/FBN1_tableS1_allmissense.tsv")
df.read_data()
df.create_filters()

stored_df_data = df.data.copy(deep = True)    

df_test = ps.DataProcessor("data/FBN1_tableS1_allmissense.tsv")
df_test.read_data()
df_test.create_filters()
    
stored_df_test = df_test.data.copy(deep = True)    

tableS1_cons = pd.read_csv("data/FBN1_tableS1_indomconserved.tsv", sep='\t', low_memory=False, na_values=['.'])
tableS1 = df.data.join(tableS1_cons.set_index("variantvcf"), on="variantvcf")

for a in range(0,10):
    i += 1
    print(i)
    
    df.data = stored_df_data.copy(deep = True)
    df_test.data = stored_df_test.copy(deep = True)    
    df.create_filters()
    df_test.create_filters()
    index_train = select_random_elements(list(df.data[df.is_ctrl].index), 38, seed = a)
    df.data = df.data[~df.data.index.isin(index_train)]
    df.create_filters()
    filtering_test = ~df_test.training_sets & df_test.is_mis 
    df_test.data = df_test.data[filtering_test]
    df_test.create_filters()
    filtering_test_2 = ((df_test.is_denovo & df_test.is_case) | (df_test.data.index.isin(index_train) & df_test.is_ctrl))
    df_test.data = df_test.data[filtering_test_2]
    df_test.create_filters()
    
    data, predictors, ks_case_ctrl_bf = ps.predictors_computation(df, df.is_mis)
    predictors = [x for x in predictors if x not in ["ClinPred_Score","MVP_rankscore"]]
    del ks_case_ctrl_bf["MVP_rankscore"]

    base = {}
    base = reload(base, ~df.training_sets & df.is_mis & ~df.is_denovo, df, predictors, ks_case_ctrl_bf, True)

    data, predictors, ks_case_ctrl_bf = ps.predictors_computation(df, ~df.is_cys)
    predictors = [x for x in predictors if x not in ["ClinPred_Score","MVP_rankscore"]]
    del ks_case_ctrl_bf["MVP_rankscore"]

    cyst = {}
    cyst = reload(cyst, ~df.is_cys, df, predictors, ks_case_ctrl_bf, True)

    #loading conserved sites and merging tables
    tableS1_test = df_test.data.join(tableS1_cons.set_index("variantvcf"), on="variantvcf")
    
    data, predictors, ks_case_ctrl_bf = ps.predictors_computation(df, ~df.is_cys & (tableS1["in_dom_conserved"] == 0))
    
    dom = {}
    dom = reload(dom, ~df.training_sets & df.is_mis & ~df.is_denovo & ~df.is_cys & (tableS1["in_dom_conserved"] == 0), df, predictors, ks_case_ctrl_bf, True)
    predictors = [x for x in predictors if x not in ["ClinPred_Score","MVP_rankscore"]]

    for pred in predictors:
        thresh = {pred: 0.5}
        trace_1 = build_trace(
            pd.DataFrame(df_test.data[(df_test.is_case & ~df_test.is_ctrl)][pred]),
            thresh,
            name="1",
            text=False,
            y_percentage=False)

        trace_2 = build_trace(
            pd.DataFrame(df_test.data[(~df_test.is_case & df_test.is_ctrl)][pred]),
            thresh,
            name="2",
            text=False,
            y_percentage=False)
    
        results_per_pred_base[0][pred] += list(metrics(trace_1, trace_2, 0))[0]
        results_per_pred_base[1][pred] += list(metrics(trace_1, trace_2, 0))[1]
        results_per_pred_base[2][pred] += list(metrics(trace_1, trace_2, 0))[2]

    for pred in predictors:
        thresh = {pred: 0.5}
        trace_1 = build_trace(
            pd.DataFrame(df_test.data[(df_test.is_case & ~df_test.is_ctrl) & (~df_test.is_cys & (tableS1_test["in_dom_conserved"] == 0))][pred]),
            thresh,
            name="1",
            text=False,
            y_percentage=False)

        trace_2 = build_trace(
            pd.DataFrame(df_test.data[(~df_test.is_case & df_test.is_ctrl) & (~df_test.is_cys & (tableS1_test["in_dom_conserved"] == 0))][pred]),
            thresh,
            name="2",
            text=False,
            y_percentage=False)
    
        results_per_pred_3[0][pred] += list(metrics(trace_1, trace_2, 0))[0]
        results_per_pred_3[1][pred] += list(metrics(trace_1, trace_2, 0))[1]
        results_per_pred_3[2][pred] += list(metrics(trace_1, trace_2, 0))[2]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Example data for other models' performance (replace with actual data)
# Assume that each column represents the performance of other models on a specific metric
other_models_performance = results_per_pred_base
# Performance of your model for each metric
gemvap1_performance = me_1_123
gemvap2_performance = me_2_123

# Convert dictionary to a list of values for boxplot
metrics_ = list(other_models_performance.columns)
data = [other_models_performance[metric_] for metric_ in metrics_]

plt.style.use('seaborn-white')
fig, ax = plt.subplots(figsize=(8, 6))

# Define softer, professional colors
box_color = 'lightsteelblue'
median_color = 'midnightblue'
point_color = 'darkgreen'
point_color_2 = 'cyan'

# Plot the boxplot for other models with softer colors
boxplot = ax.boxplot(data, patch_artist=True, labels=metrics_, 
                     boxprops=dict(facecolor=box_color, color='gray', linewidth=1.5),
                     medianprops=dict(color=median_color, linewidth=2),
                     whiskerprops=dict(color='gray', linewidth=1.5),
                     capprops=dict(color='gray', linewidth=1.5),
                     flierprops=dict(markerfacecolor='gray', marker='o', markersize=5, linestyle='none'))

# Overlay your model's performance as a soft red point
for i, metric_ in enumerate(metrics_, start=1):
    ax.scatter(i, gemvap1_performance[metric_], color=point_color, s=100, zorder=3, edgecolor='black', 
               label='GEMVAP 1' if i == 1 else "")
    ax.scatter(i, gemvap2_performance[metric_], color=point_color_2, s=100, zorder=3, edgecolor='black', 
               label='GEMVAP 2' if i == 1 else "")
first_legend = ax.legend([boxplot["boxes"][0]], 
                         ['Other models'], loc='lower right')

# Add the first legend manually to avoid being overridden by the second
ax.add_artist(first_legend)
# Customize ticks, labels, and title
ax.set_title('Performance Comparison on Full Dataset', fontsize=16, fontweight='bold', pad=15)
ax.set_ylabel("Model's Performance Values", fontsize=14, labelpad=10)
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(["F1 Score", "Matthews Correlation Coefficient", "Accuracy"], fontsize=12)
ax.tick_params(axis='y', labelsize=12)

# Remove gridlines as requested
ax.grid(False)

# Add a more visible legend with a light background
ax.legend(loc='upper center', fontsize=12, frameon=True, framealpha=0.8, facecolor='white', edgecolor='gray', 
          bbox_to_anchor=(0.5, -0.1), ncol=1, markerscale=1.5)

# Annotate your model's points
for i, metric_ in enumerate(metrics_, start=1):
    ax.annotate(f'{my_model_performance[metric_]:.2f}', 
                (i, my_model_performance[metric_]),
                textcoords="offset points", 
                xytext=(20,-3), 
                ha='center', fontsize=12, color=point_color)

# Adjust layout for better spacing
plt.tight_layout()

# Save as a high-resolution image for publication
plt.savefig('data/model_performance_comparison_full.png', dpi=300)

# Show the plot
plt.show()